# Price Deployment Model V1

This notebook is the clear workflow for the active nightly price model.

The model we deploy is the leakage-free blended price model. It combines the global baseline model and segmented price model (in the sence that i segmented the market into luxurious listings and non luxurious), and it keeps the baseline model inside the bundle as fallback.

## Cell 1 - Locate Project Files

This cell finds the deployment folder, the model artifact folder, and the training script. It works whether the notebook is opened from the repository root or directly from this folder.

In [1]:
from pathlib import Path
import json
import runpy

def find_deployment_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "code files" / "ml" / "deployment_models",
        Path.cwd().parent / "deployment_models",
    ]
    for candidate in candidates:
        if (candidate / "price_deployment_model_v1.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find price_deployment_model_v1.py")

DEPLOYMENT_DIR = find_deployment_dir()
PROJECT_ROOT = DEPLOYMENT_DIR.parents[2]
ARTIFACT_DIR = PROJECT_ROOT / "data" / "gold" / "modeling" / "deployment_models"
PRICE_SCRIPT = DEPLOYMENT_DIR / "price_deployment_model_v1.py"

print("Deployment folder:", DEPLOYMENT_DIR)
print("Artifact folder:", ARTIFACT_DIR)
print("Price script:", PRICE_SCRIPT)

Deployment folder: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models
Artifact folder: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\data\gold\modeling\deployment_models
Price script: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\price_deployment_model_v1.py


## Cell 2 - Rebuild The Price Deployment Model

Run this cell when you want to retrain and resave the active price deployment artifact. The script asks for the MySQL password using `getpass`, so the password stays hidden.

In [2]:
# This executes the tested deployment pipeline and saves the .joblib + metadata files.
# It does not use test leakage: threshold and blend choices are learned before final test evaluation.
runpy.run_path(str(PRICE_SCRIPT), run_name="__main__")

# Price Deployment Model V1

## Decision

- Use blended price deployment model over current baseline: NO
- Chosen validation blend strategy: bucket_specific_weight

## Validation strategy table

               strategy                         weight_scope  global_weight       r2      rmse       mae     mape  residual_mean  residual_std  weight_normal  weight_luxury  global_fallback_weight
 bucket_specific_weight predicted_price_bucket_from_baseline            NaN 0.677298 40.616453 28.119749 0.202027      -0.584994     40.612240            NaN            NaN                    0.74
segment_specific_weight                    predicted_segment            NaN 0.677216 40.621646 28.130083 0.201781      -0.553041     40.617882           0.75           0.25                     NaN
          global_weight                               global           0.74 0.676927 40.639810 28.104448 0.201642      -0.532921     40.636316            NaN            NaN                     NaN
    segmented_ref

{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'C:\\Users\\mario\\Documents\\DSMLAISL LEARNING PATH\\Portugal-Rental-Investment-\\code files\\ml\\deployment_models\\price_deployment_model_v1.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __b

## Cell 3 - Read The Final Scores

This cell reads the deployment metadata and displays the final test metrics for the baseline, segmented model, and selected blended model.

In [4]:
import pandas as pd

metadata_path = ARTIFACT_DIR / "nightly_price_deployment_model_v1_metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

price_metrics = pd.DataFrame(metadata["overall_test_metrics"])
price_metrics.sort_values(["r2", "rmse", "mae"], ascending=[False, True, True])

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\mario\\Documents\\DSMLAISL LEARNING PATH\\Portugal-Rental-Investment-\\data\\gold\\modeling\\deployment_models\\nightly_price_deployment_model_v1_metadata.json'

## Cell 4 - Confirm The Saved Artifact

This cell loads the `.joblib` bundle and confirms the selected model type. This is the object that later deployment code should load for nightly price prediction.

In [ ]:
import joblib

model_path = ARTIFACT_DIR / "nightly_price_deployment_model_v1.joblib"
price_bundle = joblib.load(model_path)

print("Artifact:", model_path)
print("Bundle type:", price_bundle.get("bundle_type"))
print("Selected model:", price_bundle.get("selected_model"))
print("Feature count:", len(price_bundle.get("feature_columns", [])))

## Cell 5 - Deployment Notes

The active price model is selected because it beats the baseline on R2, RMSE, and MAE. The CSV files from testing are not required for deployment because the report, registry, metadata, and artifact contain the important information.